# 02 · Limpieza, outliers y agregación

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Detectar **outliers** físicos (errores de sensor) y discutir cuándo conservarlos.
2. Practicar **imputación** de huecos cortos y reconocer cuándo *no* rellenar.
3. Hacer **agregación temporal** con la regla correcta para cada variable (lluvia → suma, caudal → media o máximo).

**Datasets.** Mismos que el notebook 01.

In [ ]:
from itertools import groupby
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import median_abs_deviation

from cst import datos as ud

plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.grid': True, 'grid.alpha': 0.3})

caudal   = ud.cargar_caudal_genil()
lluvia_m = ud.cargar_lluvia_genil()
piezo    = ud.cargar_piezometria()

## 1 · Outliers en caudal

Tres métodos clásicos. Usamos un periodo de cobertura buena (1975–2020) para que la mediana sea representativa.

In [ ]:
q = caudal.loc['1975':'2020'].copy()
x = q.dropna().values

# 1.a — z-score robusto (mediana + MAD)
mediana = np.median(x)
mad = median_abs_deviation(x, scale='normal')   # equivalente al sigma gaussiano
z = (q - mediana) / mad
out_mad = z.abs() > 3.5

# 1.b — IQR
q1, q3 = np.quantile(x, [0.25, 0.75])
iqr = q3 - q1
out_iqr = (q < q1 - 1.5*iqr) | (q > q3 + 1.5*iqr)

# 1.c — Hampel (ventana móvil de 30 días)
ventana = 30
med_movil = q.rolling(ventana, center=True, min_periods=10).median()
mad_movil = (q - med_movil).abs().rolling(ventana, center=True, min_periods=10).median() * 1.4826
z_hampel = (q - med_movil) / mad_movil.replace(0, np.nan)
out_hampel = z_hampel.abs() > 3.5

print(f'% outliers · z-score robusto : {100*out_mad.mean():.2f}%')
print(f'% outliers · IQR             : {100*out_iqr.mean():.2f}%')
print(f'% outliers · Hampel (30 d)   : {100*out_hampel.mean():.2f}%')

In [ ]:
# Inspección visual: ¿qué marca cada método en el evento de 1985?
ventana_evento = slice('1985-01-15', '1985-03-15')
q_v = q.loc[ventana_evento]
out_v = {
    'z-score robusto': out_mad.loc[ventana_evento],
    'IQR':             out_iqr.loc[ventana_evento],
    'Hampel':          out_hampel.loc[ventana_evento],
}

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, (nombre, mask) in zip(axes, out_v.items()):
    ax.plot(q_v.index, q_v.values, color='#1f6f8b', lw=1)
    ax.scatter(q_v.index[mask.fillna(False)], q_v[mask.fillna(False)],
               color='#c2410c', zorder=3, s=30, label='outlier')
    ax.set_ylabel(nombre); ax.legend(loc='upper right')
axes[-1].set_xlabel('Fecha')
plt.suptitle('Detección de outliers durante la crecida de feb 1985', y=1.02)
plt.tight_layout()

**Observa:** los tres métodos marcan el pico como "outlier". Pero **es real** — es justamente el evento que querríamos predecir. Por eso en hidrología:

- Nunca se eliminan outliers "a ciegas".
- Se comparan con: lluvia coincidente, estaciones vecinas, hidrograma típico.
- Si pasa la inspección, **se conservan** (y a veces se etiquetan como `evento_extremo=True` para que el modelo pueda tratarlos).

El método robusto **móvil** (Hampel) es el que mejor distingue ruido sensor (puntos aislados) de eventos (rachas coherentes), porque compara cada punto con su entorno local.

## 2 · Outliers "obvios" — valores físicamente imposibles

Antes de aplicar nada estadístico, conviene una limpieza por reglas:

In [ ]:
# Reglas físicas para el Genil:
#   - caudal < 0 → imposible
#   - caudal > 1000 m³/s → extremadamente improbable en Pinos-Genil (max histórico ~230)
regla_fisica = (caudal < 0) | (caudal > 1000)
print(f'Violaciones de regla física: {regla_fisica.sum()} de {caudal.notna().sum()} obs.')
if regla_fisica.any():
    print(caudal[regla_fisica].head())

caudal_limpio = caudal.mask(regla_fisica)
print(f'NaN antes : {caudal.isna().sum():,}')
print(f'NaN después: {caudal_limpio.isna().sum():,}')

## 3 · Datos faltantes — tipo y tamaño de gaps

Antes de imputar, miremos la **estructura** de los huecos.

In [ ]:
def tamanos_de_gaps(serie):
    """Devuelve lista con la longitud (en pasos) de cada racha de NaN."""
    mask = serie.isna().astype(int).values
    return [sum(1 for _ in g) for k, g in groupby(mask) if k == 1]

gaps = tamanos_de_gaps(caudal_limpio)
print(f'Número de gaps : {len(gaps)}')
print(f'Tamaño medio   : {np.mean(gaps):.1f} días')
print(f'Tamaño mediano : {int(np.median(gaps))} días')
print(f'Tamaño máx     : {max(gaps)} días')
print(f'Gaps de 1 día  : {sum(g == 1 for g in gaps)} ({100*sum(g==1 for g in gaps)/len(gaps):.1f}%)')
print(f'Gaps > 30 días : {sum(g > 30 for g in gaps)}')

In [ ]:
# Histograma de tamaños de gap (escala log)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(gaps, bins=np.logspace(0, np.log10(max(gaps)+1), 40),
        color='#7c3aed', alpha=0.8)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Longitud del gap (días, escala log)')
ax.set_ylabel('Nº de gaps (log)')
ax.set_title('Distribución del tamaño de huecos en el caudal')
plt.tight_layout()

## 4 · Imputación: ¿cuándo sí y cuándo no?

Vamos a comparar dos estrategias en un periodo con gaps cortos.

In [ ]:
# Escogemos un periodo razonablemente cubierto y simulamos un hueco corto adicional
tramo = caudal_limpio.loc['2010-01-01':'2010-03-31'].copy()
tramo_real = tramo.copy()
tramo.loc['2010-02-10':'2010-02-13'] = np.nan   # gap simulado de 4 días

estrategias = {
    'forward-fill': tramo.ffill(),
    'interpolación lineal': tramo.interpolate('linear'),
    'interpolación spline orden 3': tramo.interpolate('spline', order=3),
}

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tramo_real.index, tramo_real.values, color='black', lw=1, label='real (referencia)')
for nombre, serie_imp in estrategias.items():
    ax.plot(serie_imp.index, serie_imp.values, lw=1.4, ls='--', label=nombre)
ax.axvspan('2010-02-10', '2010-02-13', color='red', alpha=0.1)
ax.set_ylabel('Q (m³/s)'); ax.legend()
ax.set_title('Imputación de un gap simulado de 4 días')
plt.tight_layout()

# Errores en el tramo imputado
for nombre, serie_imp in estrategias.items():
    err = (serie_imp.loc['2010-02-10':'2010-02-13'] - tramo_real.loc['2010-02-10':'2010-02-13']).abs().mean()
    print(f'MAE en el gap · {nombre:30s}: {err:.3f} m³/s')

### Cuándo **no** rellenar

- Gaps **largos** (> ~30 días) en variables dinámicas: la imputación introduce sesgo y artefactos.
- **Piezometría**: gaps de meses son la norma, no la excepción. Mantén los NaN.
- Si la variable tiene **eventos** (crecidas, sequías) más cortos que el gap.

Demostración con piezometría:

In [ ]:
# Reindexar a frecuencia regular para ver la pinta del gap
piezo_diaria = piezo.reindex(pd.date_range(piezo.index.min(), piezo.index.max(), freq='D'))

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(piezo_diaria.index, piezo_diaria.values, marker='o', ms=2, lw=0.5, color='#0d9488')
axes[0].set_ylabel('Cota (m s.n.m.)'); axes[0].set_title('Original (con NaN)')

axes[1].plot(piezo_diaria.interpolate('linear').index,
             piezo_diaria.interpolate('linear').values, color='#dc2626', lw=0.5)
axes[1].plot(piezo_diaria.index, piezo_diaria.values, 'o', ms=2, color='#0d9488')
axes[1].set_ylabel('Cota (m s.n.m.)')
axes[1].set_title('Interpolación lineal — fíjate que está "inventando" años enteros')
plt.tight_layout()

## 5 · Agregación temporal — la regla depende de la variable

`pandas.DataFrame.resample` reescribe el índice a una nueva frecuencia. Lo crítico es **qué función aplicar**:

| Variable | Regla |
|---|---|
| caudal medio | `mean()` |
| caudal máximo (eventos) | `max()` |
| lluvia acumulada | `sum()` |
| nivel piezométrico | `mean()` o `median()` |

Usar la regla equivocada cambia físicamente la variable.

In [ ]:
q_d = caudal_limpio.loc['2010':'2020']
q_mensual_media = q_d.resample('MS').mean()
q_mensual_max   = q_d.resample('MS').max()

fig, ax = plt.subplots()
ax.plot(q_d.index, q_d.values, color='#1f6f8b', lw=0.4, alpha=0.6, label='diario')
ax.plot(q_mensual_media.index, q_mensual_media.values, color='#2563eb', lw=1.4,
        marker='o', ms=3, label='media mensual')
ax.plot(q_mensual_max.index, q_mensual_max.values, color='#c2410c', lw=1.4,
        marker='^', ms=3, label='máx mensual')
ax.set_ylabel('Q (m³/s)'); ax.legend()
plt.tight_layout()

In [ ]:
# Si tuviéramos lluvia diaria, la agregación correcta sería suma
# Aquí la fuente ROEA ya nos da lluvia mensual; lo simulamos para enseñar la trampa.
rng = np.random.default_rng(0)
lluvia_diaria_sim = pd.Series(
    rng.gamma(0.2, 4.0, size=len(q_d)),
    index=q_d.index, name='lluvia_sim_mm'
)
lluvia_diaria_sim[lluvia_diaria_sim < 0.1] = 0

media = lluvia_diaria_sim.resample('MS').mean()
suma  = lluvia_diaria_sim.resample('MS').sum()

fig, ax = plt.subplots()
ax.plot(media.index, media.values, label='media (¡incorrecto para lluvia!)', color='#dc2626')
ax.plot(suma.index, suma.values, label='suma (correcto)', color='#2563eb')
ax.set_ylabel('Lluvia mensual (mm)'); ax.legend()
ax.set_title('Lluvia: la regla correcta es suma')
plt.tight_layout()

## 6 · Ejercicios

1. **Hampel sensible.** Vuelve a aplicar el filtro Hampel con ventanas de 7, 30 y 90 días. ¿Cómo cambia el porcentaje de outliers? ¿Cuál "protege" mejor el pico de 1985?
2. **Política de imputación.** Define una función `imputar_caudal(serie, gap_max=3)` que rellene gaps **estrictamente menores** que `gap_max` días con interpolación lineal y deje el resto como NaN.
3. **Caudal máximo anual.** Calcula el caudal máximo de cada año hidrológico (oct→sep) y dibuja la serie. ¿Hay tendencia visible?
4. **Reto.** Para `PZ0267014`, propón una manera *defendible* de pasar a frecuencia mensual sin imputar agresivamente. Pista: `resample('MS').mean()` sobre los puntos disponibles, pero descartando meses sin medición. ¿Cuántos meses quedan?
5. **Outliers en lluvia diaria.** Aplica los métodos genéricos (IQR, MAD, Hampel) a la lluvia diaria del SAIH (`ud.cargar_lluvia_genil()`). ¿Qué fracción de días marca cada método como outlier? Mira los valores marcados — ¿son lluvia real o errores? Discute por qué los métodos pensados para variables aproximadamente simétricas fallan en una distribución con **muchos ceros + cola pesada**. Propón una alternativa (transformación log(1+P), umbral por cuantil P99.9, o regla física tipo "P > 200 mm/día es sospechoso").